This is what ChatGPT generated for me, so I will leave it here, but it will not work now.

This downloads the data from GSE2034 as a csv.

I created this folder to download the csv /content/GSE2034

In [1]:
import csv
import gzip
import re
import urllib.request
from pathlib import Path
import os


# The csv files will be saved in the root_folder directory by the job.slurm script.
root_folder = os.path.abspath("")
print(root_folder)


/home/antoniodelagarza01/dl6379/TabPFN_Project/runs/v01_151589


Test run for using GSE2034 data for train and test

In [2]:
# Authentication code - No need to read it!
import os
import getpass

# --- Prior Labs Authentication ---
print("Prior Labs Authentication")
print(
    "\nTabPFN model weights or API access via client requires a free "
    "Prior Labs account and acceptance of the non-commercial "
    "license.\n"
)

tabpfn_token = "tabpfn_sk_NIelfQ4E3hlqo9mDC99R2QOY-mO75MIgMY_ST7CX9Eg"

# 1. Try Colab secret TABPFN_TOKEN
try:
    from google.colab import userdata
    tabpfn_token = userdata.get("TABPFN_TOKEN")
    if tabpfn_token:
        os.environ["TABPFN_TOKEN"] = tabpfn_token
        console.print("Found TABPFN_TOKEN in Colab secrets.")
except Exception:
    pass

# 2. If no token found, prompt the user
if not tabpfn_token:
    print(
            "To get your access token:\n\n"
            "  1. Go to ux.priorlabs.ai and sign up / log in\n"
            "  2. Accept the license at ux.priorlabs.ai/account/licenses\n"
            "  3. Copy your Access Token from ux.priorlabs.ai/account\n\n"
            "Tip: Save the token as a Colab secret named "
            "TABPFN_TOKEN to skip this step next time."

    )
    while not tabpfn_token:
        token_input = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ")
        if token_input.strip():
            tabpfn_token = token_input.strip()
            os.environ["TABPFN_TOKEN"] = tabpfn_token
        else:
            print("Token cannot be empty. Please try again.")

else:
    os.environ["TABPFN_TOKEN"] = tabpfn_token

print("TabPFN Demo: Backend Selection")
print("\nThis script can run TabPFN using one of two backends:")
print("  local: Uses a local GPU (NVIDIA). Requires CUDA.")
print(
    "  client: Uses the TabPFN API. Requires an internet connection and a free account."
)

backend = "local"
while backend is None:
  print(
      "\nChoose your backend: - If no text box is shown, restart the cell.",
  )
  user_input = input("Enter 'local' or 'client' and press return:")
  if user_input not in ["local", "client"]:
    continue
  backend = user_input

print(
    f"\nYou have selected the '{backend}' backend."
)

print(f"Setting up {backend} backend")

if backend == "local":
    print("Attempting local backend setup...")
    import torch

    if not torch.cuda.is_available():
        print(
            "Error: GPU device not found. For fast training, please enable GPU."
        )
        print(
            "In Colab: Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU."
                    )
        raise SystemError("GPU device not found.")
    print("GPU is available.")




    print("")
    print("Importing local TabPFN library...")

    from tabpfn import TabPFNClassifier, TabPFNRegressor
    from tabpfn.finetuning import FinetunedTabPFNClassifier

    print("TabPFN (local) imported successfully.")
elif backend == "client":
    print("Attempting client backend setup...")
    print("Importing TabPFN client library...")
    from tabpfn_client import TabPFNClassifier, TabPFNRegressor, init

    init()
    print(" TabPFN (client) initialized.")


Prior Labs Authentication

TabPFN model weights or API access via client requires a free Prior Labs account and acceptance of the non-commercial license.

TabPFN Demo: Backend Selection

This script can run TabPFN using one of two backends:
  local: Uses a local GPU (NVIDIA). Requires CUDA.
  client: Uses the TabPFN API. Requires an internet connection and a free account.

You have selected the 'local' backend.
Setting up local backend
Attempting local backend setup...


GPU is available.

Importing local TabPFN library...


/home/antoniodelagarza01/myEnvs/dl6379_TabPFN/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


TabPFN (local) imported successfully.


Pre-trained TabPFN

In [3]:
# This only runs locally as there is no fine tunning model in client

import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split


# Initialize the fine-tuning classifier
tabpfn_classifier = TabPFNClassifier(random_state=42, n_estimators=4)

# Change this to your desired path
df = pd.read_csv(f"{root_folder}/GSE2034_samples_by_probes.csv", index_col=0)

# Expression CSVs: patient IDs in the first column, probe IDs as other columns.
x = df.drop(["relapse (1=True)"], axis=1).astype(float)
# x_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_expression.csv", index_col=0)

y = df["relapse (1=True)"].astype(int)
# y_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_labels.csv", index_col=0)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)

# This will select the k best probes based on the F-test for classification
# As the data contains 22K+ probes, we select the top 500 most relevant ones for classification for better results
# k=150 gave the best accuracy at 68%
selector = SelectKBest(score_func=f_classif, k=150)
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

print("Shape of data:\n")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}\n")

# Train and evaluate the TabPFN classifier
print("Training TabPFN classifier...")
tabpfn_classifier.fit(X_train, y_train)
predictions = tabpfn_classifier.predict(X_test)
print(f"predictions (shape {predictions.shape}): {predictions}")
print(f"actuals (shape {y_test.shape}): {y_test}")
probas = tabpfn_classifier.predict_proba(X_test)
print(f"probabilities (shape {probas.shape}): {probas}")
accuracy = tabpfn_classifier.score(X_test, y_test)
print(f"Test accuracy: {accuracy}")

Shape of data:

X_train: (228, 150)
X_test: (58, 150)
y_train: (228,)
y_test: (58,)

Training TabPFN classifier...


predictions (shape (58,)): [1 0 0 1 0 0 0 1 0 0 0 1 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 0 0
 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1]
actuals (shape (58,)): sample
GSM36786    0
GSM37044    0
GSM36920    1
GSM36989    1
GSM37004    1
GSM36932    0
GSM37060    0
GSM36850    0
GSM36973    1
GSM36810    0
GSM36962    0
GSM36901    0
GSM36923    1
GSM37042    1
GSM36782    0
GSM36997    1
GSM37010    0
GSM37028    1
GSM36822    0
GSM37045    0
GSM36947    1
GSM36833    0
GSM36977    0
GSM36856    0
GSM36888    1
GSM37052    1
GSM37051    1
GSM36994    1
GSM36886    0
GSM36819    0
GSM36980    0
GSM36861    0
GSM37046    0
GSM36870    1
GSM36823    0
GSM36981    0
GSM36840    0
GSM36852    0
GSM37017    0
GSM36941    1
GSM36958    0
GSM36807    0
GSM36799    0
GSM36801    0
GSM36952    1
GSM36854    0
GSM36924    1
GSM36998    1
GSM37026    1
GSM36869    0
GSM36984    0
GSM37011    1
GSM36954    1
GSM36988    0
GSM36783    0
GSM36893    0
GSM36970    0
GSM36837    0
Name: relaps

probabilities (shape (58, 2)): [[0.26317444 0.7368256 ]
 [0.93178666 0.06821331]
 [0.6997942  0.30020583]
 [0.3809805  0.6190195 ]
 [0.68832785 0.3116722 ]
 [0.7970687  0.2029313 ]
 [0.82731473 0.17268525]
 [0.47293028 0.5270697 ]
 [0.6692537  0.3307463 ]
 [0.98163855 0.01836148]
 [0.7283888  0.2716112 ]
 [0.47739983 0.5226002 ]
 [0.38301203 0.61698794]
 [0.96082693 0.03917305]
 [0.8467063  0.1532937 ]
 [0.13440108 0.8655989 ]
 [0.9048121  0.09518787]
 [0.7246681  0.2753319 ]
 [0.98160595 0.01839408]
 [0.75551325 0.24448672]
 [0.73836756 0.26163244]
 [0.956562   0.04343804]
 [0.23036574 0.76963425]
 [0.820634   0.179366  ]
 [0.5082771  0.4917229 ]
 [0.7478024  0.25219762]
 [0.6362843  0.36371574]
 [0.7397608  0.26023924]
 [0.58675694 0.41324306]
 [0.23491272 0.7650873 ]
 [0.9740055  0.02599445]
 [0.4881642  0.5118358 ]
 [0.94824564 0.05175435]
 [0.36697927 0.6330207 ]
 [0.82776177 0.17223823]
 [0.6792043  0.32079574]
 [0.8014089  0.19859111]
 [0.8017196  0.19828036]
 [0.46880788 0.5311

Test accuracy: 0.5862068965517241


Fine-tunned TabPFN

In [4]:
# This only runs locally as there is no fine tunning model in client

import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from tabpfn.finetuning import FinetunedTabPFNClassifier


# Initialize the fine-tuning classifier
tabpfn_classifier = FinetunedTabPFNClassifier(
    epochs=100,               # Default is 30 (range: 10 to 100)
    learning_rate=1e-5,                 # Learning rate for weight updates
    # use_lr_scheduler=True,  # warmup + cosine decay
    lr_warmup_only=True,
    eval_metric="log_loss",
    early_stopping_patience=15,
    n_finetune_ctx_plus_query_samples=50000,
    n_estimators_finetune=4,
    n_estimators_validation=4,
    n_estimators_final_inference=16,
    device='cuda',           # Strongly recommended to use a GPU ('cuda' or 'mps')
    random_state=42
)

# Change this to your desired path
df = pd.read_csv(f"{root_folder}/GSE2034_samples_by_probes.csv", index_col=0)

# Expression CSVs: patient IDs in the first column, probe IDs as other columns.
x = df.drop(["relapse (1=True)"], axis=1).astype(float)
# x_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_expression.csv", index_col=0)

y = df["relapse (1=True)"].astype(int)
# y_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_labels.csv", index_col=0)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)

# This will select the k best probes based on the F-test for classification
# As the data contains 22K+ probes, we select the top 500 most relevant ones for classification for better results
# k=150 gave the best accuracy at 68%
selector = SelectKBest(score_func=f_classif, k=150)
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

print("Shape of data:\n")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}\n")

# Train and evaluate the TabPFN classifier
print("Training TabPFN classifier...")
tabpfn_classifier.fit(X_train, y_train)
predictions = tabpfn_classifier.predict(X_test)
print(f"predictions (shape {predictions.shape}): {predictions}")
print(f"actuals (shape {y_test.shape}): {y_test}")
probas = tabpfn_classifier.predict_proba(X_test)
print(f"probabilities (shape {probas.shape}): {probas}")
accuracy = tabpfn_classifier.score(X_test, y_test)
print(f"Test accuracy: {accuracy}")

/home/antoniodelagarza01/myEnvs/dl6379_TabPFN/lib64/python3.9/site-packages/tabpfn/finetuning/finetuned_classifier.py:161: UserWarning: `use_fixed_preprocessing_seed` should only be used if `n_estimators_finetune` == `n_estimators_validation` == `n_estimators_final_inference`. Consider setting the number of estimators for validation and final inference to the same value as `n_estimators_finetune`(=4).
  super().__init__(


Shape of data:

X_train: (228, 150)
X_test: (58, 150)
y_train: (228,)
y_test: (58,)

Training TabPFN classifier...


/home/antoniodelagarza01/myEnvs/dl6379_TabPFN/lib64/python3.9/site-packages/tabpfn/finetuning/finetuned_classifier.py:414: UserWarning: `output_dir` is not set. This means no checkpointing will be done and all progress will be lost if the training is interrupted.
  super().fit(X, y, X_val=X_val, y_val=y_val, output_dir=output_dir)


Finetuning Epoch 1/100:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 2/100:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 3/100:   0%|          | 0/1 [00:00<?, ?it/s]

... (46 outputs left out)

Finetuning Epoch 50/100:   0%|          | 0/1 [00:00<?, ?it/s]

predictions (shape (58,)): [1 0 1 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 1 0 1 0 0 0
 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 1 0 0 0 0 1]
actuals (shape (58,)): sample
GSM36786    0
GSM37044    0
GSM36920    1
GSM36989    1
GSM37004    1
GSM36932    0
GSM37060    0
GSM36850    0
GSM36973    1
GSM36810    0
GSM36962    0
GSM36901    0
GSM36923    1
GSM37042    1
GSM36782    0
GSM36997    1
GSM37010    0
GSM37028    1
GSM36822    0
GSM37045    0
GSM36947    1
GSM36833    0
GSM36977    0
GSM36856    0
GSM36888    1
GSM37052    1
GSM37051    1
GSM36994    1
GSM36886    0
GSM36819    0
GSM36980    0
GSM36861    0
GSM37046    0
GSM36870    1
GSM36823    0
GSM36981    0
GSM36840    0
GSM36852    0
GSM37017    0
GSM36941    1
GSM36958    0
GSM36807    0
GSM36799    0
GSM36801    0
GSM36952    1
GSM36854    0
GSM36924    1
GSM36998    1
GSM37026    1
GSM36869    0
GSM36984    0
GSM37011    1
GSM36954    1
GSM36988    0
GSM36783    0
GSM36893    0
GSM36970    0
GSM36837    0
Name: relaps

probabilities (shape (58, 2)): [[0.35726497 0.64273506]
 [0.93250334 0.06749664]
 [0.3720703  0.6279297 ]
 [0.38052234 0.6194777 ]
 [0.81689453 0.18310547]
 [0.80214816 0.19785182]
 [0.88134766 0.11865234]
 [0.63003665 0.36996338]
 [0.61978024 0.3802198 ]
 [0.989439   0.01056102]
 [0.74121094 0.25878906]
 [0.53222656 0.46777344]
 [0.4969475  0.50305253]
 [0.96951944 0.03048055]
 [0.87109375 0.12890625]
 [0.13448866 0.86551136]
 [0.9403149  0.0596851 ]
 [0.82216525 0.17783473]
 [0.98227197 0.01772801]
 [0.89198095 0.10801904]
 [0.5756836  0.4243164 ]
 [0.9837066  0.0162934 ]
 [0.28923604 0.710764  ]
 [0.93982667 0.06017332]
 [0.44300708 0.55699295]
 [0.7138672  0.2861328 ]
 [0.5135465  0.4864535 ]
 [0.8548761  0.14512388]
 [0.6510379  0.34896216]
 [0.24862687 0.7513731 ]
 [0.96053714 0.03946284]
 [0.47753906 0.52246094]
 [0.96217835 0.03782167]
 [0.27081808 0.72918195]
 [0.93066406 0.06933594]
 [0.7736264  0.22637363]
 [0.80009764 0.19990237]
 [0.6723633  0.32763672]
 [0.6254883  0.3745

Test accuracy: 0.6896551724137931
